Lawrence Fung and Anthony Chu

# Idea/Instructions:

2) Do fine tuning of ESMfold or ColabFold, as described below.

Use Meta's ESM-2 model, which is available in ColabFold:
https://github.com/facebookresearch/esm
Links to an external site.
You can use any fasta dataset of amino acid sequences that interests you, or you can ask the professor to provide one.
The professor can provide you a fasta
Download fasta file of toxin-antitoxin protein (amino acid) sequences, on which you can predict all the protein structures.

***Link to dataset (in presentation referred to as subset II):***

https://drive.google.com/drive/folders/1oFLx4XFvv6F10H-44BU2XcksEzsK-edD?usp=drive_link

Contains:
- *TASmania_hits_seqtk-short.faa* (for code block: **Upload .faa file (fasta file)**)
- *pdb_output_dir.zip* (for Option 1 code block: **Upload .zip File**)

We used original dataset TASmania_hits_seqtk.faa from Canvas Course Project -- Spring 2025 and preprocessed it to become *TASmania_hits_seqtk-short.faa*.

*"The professor can provide you a fasta Download fasta file of toxin-antitoxin protein (amino acid) sequences, on which you can predict all the protein structures."*

# Data Used

(in presentation referred to as subset I)

OLD way:

TASmania_hits_seqtk.faa but we took $1/4$ of this file.

The commands used to create this file are:

```
$ grep -c "^>" TASmania_hits_seqtk.faa
440

$ awk 'BEGIN{i=0} /^>/{i++} i%4==1' TASmania_hits_seqtk.faa > TASmania_hits_seqtk_25pct.faa

$grep -c "^>" TASmania_hits_seqtk_25pct.faa
110
```

(in presentation referred to as subset II)

NEW: This is what we did to your original dataset TASmania_hits_seqtk.faa to become *TASmania_hits_seqtk-short.faa*


To get only sequences <= 128 amino acids
and pipe into new file *TASmania_hits_seqtk-short.faa*

```
awk '
  /^>/ {
    if (header && length(seq) <= 128)
      print header "\n" seq
    header = $0
    seq = ""
    next
  }
  { seq = seq $0 }
  END {
    if (header && length(seq) <= 128)
      print header "\n" seq
  }
' TASmania_hits_seqtk.faa > TASmania_hits_seqtk-short.faa
```

Check count of sequences in the new file

```
grep -c "^>" TASmania_hits_seqtk-short.faa

105
```

# Upload .faa file (fasta file)

In [ ]:

file_used = "TASmania_hits_seqtk-short.faa"

# upload the above file into this colab notebook
from google.colab import files
uploaded = files.upload()


Saving TASmania_hits_seqtk-short.faa to TASmania_hits_seqtk-short (1).faa


# Steps to Completion:

- Streamline getting sequences from FASTA file -> Done
- Run ESMFold to get base PDB Structures to compare against our finetuned -> Done
- Compute and save pLDDT score -> Done
- Fine tune ESM model -> TODO
- Run and Compare fine tuned ESM Model and baseline results using pLDDT


In [ ]:
!pip uninstall -y fair-esm openfold deepspeed
!pip install -U transformers accelerate sentencepiece biotite
!pip install biopython

Sources referenced:

https://huggingface.co/facebook/esmfold_v1

https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/protein_folding.ipynb


Code reference from:
https://github.com/facebookresearch/esm?tab=readme-ov-file#esmfold-structure-prediction-

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import EsmForProteinFolding, AutoTokenizer, EsmTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# check available memory before loading
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Memory already used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1")

model = model.to(device)
model.eval()

torch: 2.10.0+cu128
cuda available: True
NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 102.0 GB
Memory already used: 14.10 GB


Loading weights:   0%|          | 0/4498 [00:00<?, ?it/s]

[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.bias   | MISSING | 
esm.contact_head.regression.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmForProteinFolding(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 2560, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-35): 36 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=2560, out_features=2560, bias=True)
              (key): Linear(in_features=2560, out_features=2560, bias=True)
              (value): Linear(in_features=2560, out_features=2560, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=2560, out_features=2560, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=25

#### troubleshooting

In [ ]:
sequence = "MKTVRQERLKSIVRIL"

test_ids = tokenizer(
    sequence,
    padding=True,              # This will use ID 1 for padding, not 0
    # truncation=True,
    add_special_tokens=True,   # Crucial: adds <cls> and <eos>
    return_tensors="pt"
)["input_ids"]

# Verify the first sequence
# print("New Token IDs:", train_tokenized['input_ids'][0][:20])

print(f"Correct IDs for ESMFold: {test_ids}")

Correct IDs for ESMFold: tensor([[23, 12, 11, 16, 19,  1,  5,  6,  1, 10, 11, 15,  9, 19,  1,  9, 10, 25]])


In [ ]:
sequence = "MKTVRQERLKSIVRIL"

test_ids = tokenizer(
    sequence,
    padding=True,              # This will use ID 1 for padding, not 0
    # truncation=True,
    add_special_tokens=False,   # Crucial: adds <cls> and <eos>
    return_tensors="pt"
)["input_ids"]

# Verify the first sequence
# print("New Token IDs:", train_tokenized['input_ids'][0][:20])

print(f"Correct IDs for ESMFold: {test_ids}")

Correct IDs for ESMFold: tensor([[12, 11, 16, 19,  1,  5,  6,  1, 10, 11, 15,  9, 19,  1,  9, 10]])


Culprit is add_special_tokens = False (so we can get rid of 21: '<pad>', 22: '<mask>', 23: '<cls>', 24: '<sep>' tokens that will trip up ***af2_to_esm*** when training)

In [ ]:
# Check the actual vocab mapping for the letter M and K
print(f"ID for 'M': {tokenizer.convert_tokens_to_ids('M')}")
print(f"ID for 'K': {tokenizer.convert_tokens_to_ids('K')}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: {tokenizer.all_special_tokens}")

ID for 'M': 12
ID for 'K': 11
Vocab size: 27
Special tokens: ['<eos>', '<unk>', 'A', '<cls>', '<mask>']


In [ ]:
print({i: tokenizer.convert_ids_to_tokens(i) for i in range(25)})

{0: 'A', 1: 'R', 2: 'N', 3: 'D', 4: 'C', 5: 'Q', 6: 'E', 7: 'G', 8: 'H', 9: 'I', 10: 'L', 11: 'K', 12: 'M', 13: 'F', 14: 'P', 15: 'S', 16: 'T', 17: 'W', 18: 'Y', 19: 'V', 20: 'X', 21: '<pad>', 22: '<mask>', 23: '<cls>', 24: '<sep>'}


#### Parsing the Sequences

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import os

# this block is for getting each sequences from prof's fasta (TASmania_hits_seqtk.faa) file and doing what we've done above to each one, and then extracting them out into this pdb_output_dir
os.makedirs("pdb_output_dir", exist_ok=True)


# **Just use Option 1**

I have downloaded all these .pdb files so we can skip this block that creates the baseline .pdb files. We can simply upload the .pdb files into pdb_output_dir


# Option 1:
* When running this file, just upload pdb_output_dir with the pdb files

# Upload .zip File

In [ ]:
# Option 1: upload the zip file
pdb_directory = "content/pdb_output_dir"

from google.colab import files
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
!unzip "$uploaded_filename"


Saving pdb_output_dir.zip to pdb_output_dir.zip
Archive:  pdb_output_dir.zip
   creating: content/pdb_output_dir/
  inflating: content/pdb_output_dir/Ga0308414_1000001121.pdb  
  inflating: content/pdb_output_dir/Ga0308414_1000010105.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10145565.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10044307.pdb  
  inflating: content/pdb_output_dir/Ga0308414_101007912.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10003439.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10235192.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10001946.pdb  
  inflating: content/pdb_output_dir/Ga0308414_100028425.pdb  
  inflating: content/pdb_output_dir/Ga0308414_100049413.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10003791.pdb  
  inflating: content/pdb_output_dir/Ga0308414_100372317.pdb  
  inflating: content/pdb_output_dir/Ga0308414_10001867.pdb  
  inflating: content/pdb_output_dir/Ga0308414_100047615.pdb  
  inflating: content/pd

## Option 2 (only if you didn't do Option 1 !!):
* rerun model to predict the pdb files
* The end of Option 2 code also will download these files as a zip.

In [ ]:
"""
# this is for professor to run code, or someone without these files
# Option 2: rerun to baseline predictions to get pdb files
pdb_directory = "pdb_output_dir"

from Bio import SeqIO
import os

# this block is for getting each sequences from prof's fasta (TASmania_hits_seqtk.faa) file and doing what we've done above to each one, and then extracting them out into this pdb_output_dir
os.makedirs("pdb_output_dir", exist_ok=True)

# TODO currently need to upload each time:
# uploaded this .faa file from the prof, not sure if each person has to upload each time? how can we both have it, or do we just need to upload each time we run collab?


fasta_path = file_used
# fasta_path = "test.faa" # this is a test file


for record in SeqIO.parse(fasta_path, "fasta"):
  # print(record)
  seq_id = record.id
  sequence = str(record.seq)

  with torch.no_grad():
        pdb_string = model.infer_pdb(sequence)

  out_path = f"pdb_output_dir/{seq_id}.pdb"
  with open(out_path, "w") as f:
        f.write(pdb_string)

  print(f"saved {out_path}")

# download these pdb files for later use
# so we can skip this block in the future
!zip -r /content/pdb_output_dir.zip /content/pdb_output_dir/
from google.colab import files
files.download('pdb_output_dir.zip')
"""


'\n# this is for professor to run code, or someone without these files\n# Option 2: rerun to baseline predictions to get pdb files\npdb_directory = "pdb_output_dir"\n\nfrom Bio import SeqIO\nimport os\n\n# this block is for getting each sequences from prof\'s fasta (TASmania_hits_seqtk.faa) file and doing what we\'ve done above to each one, and then extracting them out into this pdb_output_dir\nos.makedirs("pdb_output_dir", exist_ok=True)\n\n# TODO currently need to upload each time:\n# uploaded this .faa file from the prof, not sure if each person has to upload each time? how can we both have it, or do we just need to upload each time we run collab?\n\n\nfasta_path = file_used\n# fasta_path = "test.faa" # this is a test file\n\n\nfor record in SeqIO.parse(fasta_path, "fasta"):\n  # print(record)\n  seq_id = record.id\n  sequence = str(record.seq)\n\n  with torch.no_grad():\n        pdb_string = model.infer_pdb(sequence)\n\n  out_path = f"pdb_output_dir/{seq_id}.pdb"\n  with open(out_p

# Calculate the pLDDT for each pdb file

In [ ]:
import pandas as pd
import biotite.structure.io as bsio

pdb_directory = pdb_directory
results = []

print(f"{'Protein ID':<25} | {'Mean pLDDT':<10}")
print("-" * 40)
print()


# calculate the confidence score from the pdf file just created
# import biotite.structure.io as bsio
# struct = bsio.load_structure("result.pdb", extra_fields=["b_factor"])
# print(struct.b_factor.mean())  # this will be the pLDDT


for filename in os.listdir(pdb_directory):
  if filename.endswith("pdb"):
    file_path = os.path.join(pdb_directory, filename)

    try:
      struct = bsio.load_structure(file_path, extra_fields=["b_factor"])
      mean_plddt = struct.b_factor.mean() # this is pLDDT for file

      # save results into list of dictionaries
      protein_id = filename.replace(".pdb", "")
      results.append({"ID": protein_id, "pLDDT": mean_plddt})

      # results into table
      print(f"{protein_id:<25} | {mean_plddt:.2f}")

    except Exception as e:
      print(f"Error processing {filename}: {e}")


# save these results into a dataframe > .csv
df = pd.DataFrame(results)
df.to_csv("plddt_scores.csv", index=False)
print("\nSaved all scores to plddt_scores.csv")


Protein ID                | Mean pLDDT
----------------------------------------

Ga0308414_10002492        | 0.85
Ga0308414_100128614       | 0.62
Ga0308414_100386611       | 0.78
Ga0308414_10002315        | 0.26
Ga0308414_10204732        | 0.70
Ga0308414_100001387       | 0.55
Ga0308414_10070522        | 0.34
Ga0308414_10040286        | 0.82
Ga0308414_100002616       | 0.73
Ga0308414_10106083        | 0.86
Ga0308414_100000471       | 0.49
Ga0308414_100001517       | 0.80
Ga0308414_10009046        | 0.87
Ga0308414_10145566        | 0.88
Ga0308414_100009513       | 0.83
Ga0308414_10001603        | 0.89
Ga0308414_100016435       | 0.87
Ga0308414_100111113       | 0.62
Ga0308414_10003323        | 0.74
Ga0308414_10330064        | 0.83
Ga0308414_100099116       | 0.72
Ga0308414_100224414       | 0.68
Ga0308414_100047615       | 0.81
Ga0308414_10235192        | 0.89
Ga0393395_40_2902_3051    | 0.72
Ga0308414_10031088        | 0.73
Ga0308414_10108152        | 0.77
Ga0308414_10043383        | 

#### Preliminary Results Described:

The predicted local distance difference test (pLDDT) helps us measure the "per-residue measure of local confidence". These scores are scaled from 0 to 1 in ESMFold.
[EMBL-EBI](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/plddt-understanding-local-confidence/)

pLDDT > 0.7 is considered high confidence.
[ESM Atlas](https://esmatlas.com/about
)


In this small subset generated here in our results, we can see that some mean pLDDT scores are less than 0.7. We will fine tune the model to achieve mean pLDDT scores higher than 0.7 for the majority of protein predictions.


# This is the start to the Fine Tuning Step

https://github.com/facebookresearch/esm?tab=readme-ov-file#esmfold-structure-prediction-

For this step, this is our next TODO item.

We will use pLDDT -> fine tune to beat the base model using pLDDT scores from above

In [ ]:
# Current output is ID and mean pLDDT. below adds code to split the data into
# multiple columns: protein_id,	sequence_length, baseline_plddt_0_to_1,
# baseline_plddt_0_to_100, pdb_file.
# This is useful so we can cleanly track baseline results now,
# and later compare them against the fine-tuned model results

import os
import pandas as pd
import biotite.structure.io as bsio
from Bio import SeqIO

# fasta_path = "test.faa"
fasta_path = file_used # use main file get all sequence lengths
pdb_directory = pdb_directory

# Map protein IDs to sequence lengths
seq_lengths = {}
baseline_results = []
seq_sequences = {}

for record in SeqIO.parse(fasta_path, "fasta"):
    seq_lengths[record.id] = len(record.seq)
    seq_sequences[record.id] = str(record.seq)

for filename in os.listdir(pdb_directory):
    if filename.endswith(".pdb"):
        protein_id = filename.replace(".pdb", "")
        file_path = os.path.join(pdb_directory, filename)
        try:
            struct = bsio.load_structure(file_path, extra_fields=["b_factor"])
            mean_plddt = float(struct.b_factor.mean())
            baseline_results.append({
                "protein_id": protein_id,
                "sequence": seq_sequences.get(protein_id, None),
                "sequence_length": seq_lengths.get(protein_id, None),
                "baseline_plddt_0_to_1": mean_plddt,
                "baseline_plddt_0_to_100": mean_plddt * 100,
                "pdb_file": file_path
            })

        except Exception as e:
            print(f"Error while processing {filename}: {e}")

baseline_df = pd.DataFrame(baseline_results)
baseline_df.to_csv("baseline_results.csv", index=False)
baseline_df

,protein_id,sequence,sequence_length,baseline_plddt_0_to_1,baseline_plddt_0_to_100,pdb_file
0,Ga0308414_10002492,MEISQRQAAVCAALGDHRRLLLLYAMAAEPRSVTDLVRRLGISQPA...,109,0.847801,84.780117,content/pdb_output_dir/Ga0308414_10002492.pdb
1,Ga0308414_100128614,MSKDTGEIRVIKKKKGHHGHHGGAWKVAYADFVTAMMAFFLVMWIV...,83,0.615578,61.557813,content/pdb_output_dir/Ga0308414_100128614.pdb
2,Ga0308414_100386611,MSLFEIFTSINLFFNIFFLIVLIVLLVKVIKILKVLSSILEKVESI...,98,0.776471,77.647132,content/pdb_output_dir/Ga0308414_100386611.pdb
3,Ga0308414_10002315,MTDAGCWMLDAGCWMPDDGCRMPDDGCRMLDAGCWMPDNGCWMLDA...,103,0.262176,26.217562,content/pdb_output_dir/Ga0308414_10002315.pdb
4,Ga0308414_10204732,LSRELTSRRLLLADGGIRRTAVVRRVEALLVEALKITLRYKMETILL,47,0.695820,69.582011,content/pdb_output_dir/Ga0308414_10204732.pdb
...,...,...,...,...,...,...
100,Ga0308414_10059717,VAEIIYTESYIKKAKKFIKKHPDLLSQYEKTLKLLEVNPNHPSLRL...,90,0.822483,82.248322,content/pdb_output_dir/Ga0308414_10059717.pdb
101,Ga0308414_10043733,MEKQYIKYFSILSDLNRLRIYSYLLIKPDGLYVCELSNILSLPYYT...,122,0.790403,79.040282,content/pdb_output_dir/Ga0308414_10043733.pdb
102,Ga0308414_10111501,IQIKENDWIIMDIAFSSSFKRAFKKRIKNKKEIEELFWESIALFIQ...,99,0.800617,80.061684,content/pdb_output_dir/Ga0308414_10111501.pdb
103,Ga0308414_100083213,LHLDSSALIAIVMGEPGFEGLLAKLRASAPVGLAAPALVETALVLS...,128,0.853104,85.310419,content/pdb_output_dir/Ga0308414_100083213.pdb


In [ ]:
# baseline data we will compare with after we fine tune + rerun on fine tuned
# model

print("Num proteins: ", len(baseline_df))
print("Mean of baseline pLDDT: ", baseline_df["baseline_plddt_0_to_100"].mean())
print("Median of baseline pLDDT: ", baseline_df["baseline_plddt_0_to_100"].median())
print("Min of baseline pLDDT: ", baseline_df["baseline_plddt_0_to_100"].min())
print("Max of baseline pLDDT: ", baseline_df["baseline_plddt_0_to_100"].max())

Num proteins:  105
Mean of baseline pLDDT:  72.80921085617918
Median of baseline pLDDT:  75.36653386454184
Min of baseline pLDDT:  26.2175622542595
Max of baseline pLDDT:  88.90713476783691


# Filtering out low pLDDT scores < 0.7

In [ ]:
col = 'baseline_plddt_0_to_1'

# filter out pLDDT scores lower than 0.7, keep >= 0.7
dataset_df = baseline_df[baseline_df[col] >= 0.7]
dataset_df

,protein_id,sequence,sequence_length,baseline_plddt_0_to_1,baseline_plddt_0_to_100,pdb_file
0,Ga0308414_10002492,MEISQRQAAVCAALGDHRRLLLLYAMAAEPRSVTDLVRRLGISQPA...,109,0.847801,84.780117,content/pdb_output_dir/Ga0308414_10002492.pdb
2,Ga0308414_100386611,MSLFEIFTSINLFFNIFFLIVLIVLLVKVIKILKVLSSILEKVESI...,98,0.776471,77.647132,content/pdb_output_dir/Ga0308414_100386611.pdb
7,Ga0308414_10040286,VVSYTVVFTKHALKDAKKLSAAGLRPNAEKLLAILKENPFQTPPPY...,88,0.817014,81.701389,content/pdb_output_dir/Ga0308414_10040286.pdb
8,Ga0308414_100002616,MEQQNYIPPQRQFKSLTIGDWLITFLIQAIPVVGFIMLFVWAFGGD...,93,0.726201,72.620134,content/pdb_output_dir/Ga0308414_100002616.pdb
9,Ga0308414_10106083,MRIEAIPIRLIQRPLFRQNDPEKVRALMKSIQEIGLQEPIDVLEVN...,86,0.857749,85.774892,content/pdb_output_dir/Ga0308414_10106083.pdb
...,...,...,...,...,...,...
100,Ga0308414_10059717,VAEIIYTESYIKKAKKFIKKHPDLLSQYEKTLKLLEVNPNHPSLRL...,90,0.822483,82.248322,content/pdb_output_dir/Ga0308414_10059717.pdb
101,Ga0308414_10043733,MEKQYIKYFSILSDLNRLRIYSYLLIKPDGLYVCELSNILSLPYYT...,122,0.790403,79.040282,content/pdb_output_dir/Ga0308414_10043733.pdb
102,Ga0308414_10111501,IQIKENDWIIMDIAFSSSFKRAFKKRIKNKKEIEELFWESIALFIQ...,99,0.800617,80.061684,content/pdb_output_dir/Ga0308414_10111501.pdb
103,Ga0308414_100083213,LHLDSSALIAIVMGEPGFEGLLAKLRASAPVGLAAPALVETALVLS...,128,0.853104,85.310419,content/pdb_output_dir/Ga0308414_100083213.pdb


In [ ]:
# baseline data we will compare with after we fine tune + rerun on fine tuned
# model

print("Filtering out low pLDDT scores <= 0.7 & sequences > 128")
print("Before length filter:", len(dataset_df))

# cuda out of memory issue so we need to reduce memory: keep only short sequences
dataset_df = dataset_df[dataset_df["sequence"].str.len() <= 128].copy()
print("After length filter:", len(dataset_df))

print("Num proteins: ", len(dataset_df))
print("Mean of baseline pLDDT: ", dataset_df["baseline_plddt_0_to_100"].mean())
print("Median of baseline pLDDT: ", dataset_df["baseline_plddt_0_to_100"].median())
print("Min of baseline pLDDT: ", dataset_df["baseline_plddt_0_to_100"].min())
print("Max of baseline pLDDT: ", dataset_df["baseline_plddt_0_to_100"].max())

Filtering out low pLDDT scores <= 0.7 & sequences > 128
Before length filter: 76
After length filter: 76
Num proteins:  76
Mean of baseline pLDDT:  79.13806820158545
Median of baseline pLDDT:  79.13374547498691
Min of baseline pLDDT:  70.79591836734694
Max of baseline pLDDT:  88.90713476783691


Drop unnecessary colunmns for finetuning later

In [ ]:
dataset_df = dataset_df.drop(columns=['sequence_length','baseline_plddt_0_to_100'])
dataset_df

,protein_id,sequence,baseline_plddt_0_to_1,pdb_file
0,Ga0308414_10002492,MEISQRQAAVCAALGDHRRLLLLYAMAAEPRSVTDLVRRLGISQPA...,0.847801,content/pdb_output_dir/Ga0308414_10002492.pdb
2,Ga0308414_100386611,MSLFEIFTSINLFFNIFFLIVLIVLLVKVIKILKVLSSILEKVESI...,0.776471,content/pdb_output_dir/Ga0308414_100386611.pdb
7,Ga0308414_10040286,VVSYTVVFTKHALKDAKKLSAAGLRPNAEKLLAILKENPFQTPPPY...,0.817014,content/pdb_output_dir/Ga0308414_10040286.pdb
8,Ga0308414_100002616,MEQQNYIPPQRQFKSLTIGDWLITFLIQAIPVVGFIMLFVWAFGGD...,0.726201,content/pdb_output_dir/Ga0308414_100002616.pdb
9,Ga0308414_10106083,MRIEAIPIRLIQRPLFRQNDPEKVRALMKSIQEIGLQEPIDVLEVN...,0.857749,content/pdb_output_dir/Ga0308414_10106083.pdb
...,...,...,...,...
100,Ga0308414_10059717,VAEIIYTESYIKKAKKFIKKHPDLLSQYEKTLKLLEVNPNHPSLRL...,0.822483,content/pdb_output_dir/Ga0308414_10059717.pdb
101,Ga0308414_10043733,MEKQYIKYFSILSDLNRLRIYSYLLIKPDGLYVCELSNILSLPYYT...,0.790403,content/pdb_output_dir/Ga0308414_10043733.pdb
102,Ga0308414_10111501,IQIKENDWIIMDIAFSSSFKRAFKKRIKNKKEIEELFWESIALFIQ...,0.800617,content/pdb_output_dir/Ga0308414_10111501.pdb
103,Ga0308414_100083213,LHLDSSALIAIVMGEPGFEGLLAKLRASAPVGLAAPALVETALVLS...,0.853104,content/pdb_output_dir/Ga0308414_100083213.pdb


# Split high confidence proteins >= 0.7 pLDDT into training and validataion datasets

In [ ]:
# prep data into csv for training

from Bio import SeqIO
import pandas as pd
from sklearn.model_selection import train_test_split

# fasta_path = "test.faa"
fasta_path = file_used # use main file get all sequence lengths
valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
training_rows = []

# clean up data
for record in SeqIO.parse(fasta_path, "fasta"):
    protein_id = record.id
    sequence = str(record.seq).upper()

    # if not a valid amino acid, remove it; skips invalid sequences
    if not set(sequence).issubset(valid_amino_acids):
        continue
    # in case long sequence, skip
    # if len(sequence) > 512:
    #     continue

    training_rows.append({
        "protein_id": protein_id,
        "sequence": sequence,
        "sequence_length": len(sequence)
    })

# convert cleaned data into dataframe to split and put into csv files
print("Num usable sequences: ", len(dataset_df))
print("\nHead of dataset:")
print(dataset_df.head())
print()

# split data into training and validation sets
train_df, val_df = train_test_split(
    dataset_df,
    test_size = 0.2,
    random_state = 42
)

# save training and validation datasets into csv files
train_df.to_csv("train_sequences.csv", index=False)
print("Saved train_sequences.csv")

val_df.to_csv("val_sequences.csv", index=False)
print("Saved val_sequences.csv")

print("Training size: ", len(train_df))
print("Validation size: ", len(val_df))

Num usable sequences:  76

Head of dataset:
            protein_id                                           sequence  \
0   Ga0308414_10002492  MEISQRQAAVCAALGDHRRLLLLYAMAAEPRSVTDLVRRLGISQPA...   
2  Ga0308414_100386611  MSLFEIFTSINLFFNIFFLIVLIVLLVKVIKILKVLSSILEKVESI...   
7   Ga0308414_10040286  VVSYTVVFTKHALKDAKKLSAAGLRPNAEKLLAILKENPFQTPPPY...   
8  Ga0308414_100002616  MEQQNYIPPQRQFKSLTIGDWLITFLIQAIPVVGFIMLFVWAFGGD...   
9   Ga0308414_10106083  MRIEAIPIRLIQRPLFRQNDPEKVRALMKSIQEIGLQEPIDVLEVN...   

   baseline_plddt_0_to_1                                        pdb_file  
0               0.847801   content/pdb_output_dir/Ga0308414_10002492.pdb  
2               0.776471  content/pdb_output_dir/Ga0308414_100386611.pdb  
7               0.817014   content/pdb_output_dir/Ga0308414_10040286.pdb  
8               0.726201  content/pdb_output_dir/Ga0308414_100002616.pdb  
9               0.857749   content/pdb_output_dir/Ga0308414_10106083.pdb  

Saved train_sequences.csv
Saved val_sequen

### Goal of the Finetuning

Follow this link: https://github.com/huggingface/notebooks/blob/main/examples/protein_language_modeling.ipynb

**Train on:**

sequences where pLDDT > 0.7 (high confidence predictions) — these are the "good" examples the model can learn from



**Evaluate on:**

the sequences where pLDDT was low (< 0.7) — did the finetuned model do better on those now?

# Tokenize Training Set

In [ ]:
train_df.head()

,protein_id,sequence,baseline_plddt_0_to_1,pdb_file
15,Ga0308414_10001603,MEIKFKSNKLEKSLTIPREISRTYGTMAKLVNQRMKEFLASRNLYV...,0.889071,content/pdb_output_dir/Ga0308414_10001603.pdb
11,Ga0308414_100001517,MMAFMALFWIVIVVLAVWLVIYLTRRAAGSSSQQSALDILNQRYAR...,0.801310,content/pdb_output_dir/Ga0308414_100001517.pdb
47,Ga0308414_10012106,MAEKSEIINKYQLHEKDTGSAEVQIALLTERINHLTEHFKVHVKDH...,0.811050,content/pdb_output_dir/Ga0308414_10012106.pdb
34,Ga0308414_10145565,MAKTTNLNIRVDEEVKRKAEAIFNELGLNMSTAMNMFLRYSVRYGG...,0.810977,content/pdb_output_dir/Ga0308414_10145565.pdb
43,Ga0308414_100000268,MQYAILLQRRPDGSYQASVPLLPGLTRTAATRDEVVLHLVRSDLAE...,0.736348,content/pdb_output_dir/Ga0308414_100000268.pdb


In [ ]:
val_df.head()

,protein_id,sequence,baseline_plddt_0_to_1,pdb_file
9,Ga0308414_10106083,MRIEAIPIRLIQRPLFRQNDPEKVRALMKSIQEIGLQEPIDVLEVN...,0.857749,content/pdb_output_dir/Ga0308414_10106083.pdb
49,Ga0308414_100002813,MLDTTLLLVITVTTLALIFDFINGFHDSANAIATSVLTRALSMRNA...,0.753665,content/pdb_output_dir/Ga0308414_100002813.pdb
16,Ga0308414_100016435,MSKRLNIIIEKDEHGYYAYCPDLEGCQTQGESLDEVTENIKEAIEL...,0.871481,content/pdb_output_dir/Ga0308414_100016435.pdb
0,Ga0308414_10002492,MEISQRQAAVCAALGDHRRLLLLYAMAAEPRSVTDLVRRLGISQPA...,0.847801,content/pdb_output_dir/Ga0308414_10002492.pdb
66,Ga0308414_10000411,MKGYMYILLCSDGSYYTGSTTDLERRLAQHQAGEGANHTKKRLPVK...,0.726980,content/pdb_output_dir/Ga0308414_10000411.pdb


In [ ]:
train_sequences = list(train_df['sequence']) # just the sequences
train_scores = list(train_df['baseline_plddt_0_to_1'])


val_sequences = list(val_df['sequence']) # just the sequences
val_scores = list(val_df['baseline_plddt_0_to_1'])

Tokenizing the data

In [ ]:
# train_tokenized = tokenizer(
#     train_sequences,
#     padding=True,
#     # truncation=True,
#     return_tensors="pt",
#     add_special_tokens=False # this is very important
#     )

# test_tokenized = tokenizer(
#     val_sequences,
#     padding=True,
#     # truncation=True,
#     return_tensors="pt",
#     add_special_tokens=False # this is very important
#     )

train_tokenized = tokenizer(
    train_sequences,
    padding=True,
    truncation=True,
    # max legnth since out of memory issue
    max_length=128,
    return_tensors="pt",
    add_special_tokens=False
)

test_tokenized = tokenizer(
    val_sequences,
    padding=True,
    truncation=True,
    # max legnth since out of memory issue
    max_length=128,
    return_tensors="pt",
    add_special_tokens=False
)

In [ ]:
vocab_size = tokenizer.vocab_size

train_max = train_tokenized['input_ids'].max().item()
train_min = train_tokenized['input_ids'].min().item()

print(f"Vocab Size: {vocab_size}")
print(f"Train Range: [{train_min}, {train_max}]")

Vocab Size: 26
Train Range: [0, 19]


# Dataset creation into hugging face dataset format

In [ ]:
from datasets import Dataset

# turn this data into a dataset that PyTorch can load samples from
train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 60
})

In [ ]:
# add in the pLDDT scores for as the labels
# now we have the tokenized sequences and the labels

train_dataset = train_dataset.add_column("labels", train_scores)
test_dataset = test_dataset.add_column("labels", val_scores)

train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 60
})

In [ ]:
# # doing this because HuggingFace datasets.Dataset does not work well, so make tensor dataset manually
# from torch.utils.data import TensorDataset

# train_dataset = TensorDataset(
#     train_tokenized["input_ids"],
#     train_tokenized["attention_mask"],
#     torch.tensor(train_scores) # these are the labels
# )

# test_dataset = TensorDataset(
#     test_tokenized["input_ids"],
#     test_tokenized["attention_mask"],
#     torch.tensor(val_scores) # these are the labels
# )

Low-confidence dataset Before length filter > 128

In [ ]:
# Create final low-confidence evaluation set (this is the < 0.7 group we will
# compare with after training)
# These are proteins baseline ESMFold struggled with

col = "baseline_plddt_0_to_1"

low_conf_df = baseline_df[baseline_df[col] < 0.7].copy()

print("< 0.7 confidence proteins: ", len(low_conf_df))
print("Head of low confidence set: ")
print(low_conf_df.head())

# Save for later use
low_conf_df.to_csv("low_conf_test_set.csv", index=False)
print("Saved low_conf_test_set.csv")

# tokenize, not sure if we need to
low_conf_sequences = list(low_conf_df["sequence"])
low_conf_scores = list(low_conf_df["baseline_plddt_0_to_1"])

low_conf_tokenized = tokenizer(
    low_conf_sequences,
    padding=True,
    # truncation=True,
    return_tensors="pt",
    add_special_tokens=False # this is very important
    )

from datasets import Dataset

low_conf_dataset = Dataset.from_dict(low_conf_tokenized)
low_conf_dataset = low_conf_dataset.add_column("labels", low_conf_scores)

low_conf_dataset

< 0.7 confidence proteins:  29
Head of low confidence set: 
            protein_id                                           sequence  \
1  Ga0308414_100128614  MSKDTGEIRVIKKKKGHHGHHGGAWKVAYADFVTAMMAFFLVMWIV...   
3   Ga0308414_10002315  MTDAGCWMLDAGCWMPDDGCRMPDDGCRMLDAGCWMPDNGCWMLDA...   
4   Ga0308414_10204732    LSRELTSRRLLLADGGIRRTAVVRRVEALLVEALKITLRYKMETILL   
5  Ga0308414_100001387  VQEFFYHRPEAPAAGQPSSAGVTTSRLVQWEPAIAADQRDVHAEIA...   
6   Ga0308414_10070522  LHYISIGSGKPAGRGKPGGLPYISIGSGKPAERGKPGGLHYISIGS...   

   sequence_length  baseline_plddt_0_to_1  baseline_plddt_0_to_100  \
1               83               0.615578                61.557813   
3              103               0.262176                26.217562   
4               47               0.695820                69.582011   
5               92               0.548510                54.850975   
6               75               0.343155                34.315488   

                                         pdb_file  
1  c

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 29
})

# Training configuration
https://huggingface.co/docs/transformers/training?training-args=logging

## Freeze Layers: backbone model.esm

"PyTorch not to update the weights of any layer during backpropagation. This is ideal when you only need the final layer to adapt to new classes or labels"

https://medium.com/@prabhatzade/freezing-layers-and-fine-tuning-transformer-models-in-pytorch-a-simple-guide-119cad0980c6

https://medium.com/we-talk-data/guide-to-freezing-layers-in-pytorch-best-practices-and-practical-examples-8e644e7a9598

In [ ]:
# print(model)

In [ ]:
# freeze all parameters esm
for param in model.esm.parameters():
# for param in model.model.encoder.layers[:10].parameters():
  param.requires_grad = False

# get number of elements in p for each parameter depending on T or F
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"Trainable params: {trainable:,}")
print(f"Frozen params:    {frozen:,}")

Trainable params: 692,594,594
Frozen params:    2,832,444,321


### Setting up Training

In [ ]:
from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir="./esmfold-finetuned",
#     num_train_epochs=10, # from 3 -> 10 because small dataset need more epochs
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     gradient_checkpointing=True, # save memory, but slower training
#     bf16=True, # or use fp16=True for older hardware

#     # https://www.biorxiv.org/content/10.1101/2025.02.03.635741v1.full
#     learning_rate=1e-5, # set at smaller learning rate 5e-5 -> 1e-5

#     # warmup_steps=20, # 100 -> 20 because 71 total training
#     # weight_decay=0.01,

#     logging_steps=5, # more feedback smaller dataset
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     remove_unused_columns=False,

#     # metric_for_best_model="perplexity",
#     )

# try eval strat = no and load best model at end = false
training_args = TrainingArguments(
    output_dir="./esmfold-finetuned",
    num_train_epochs=10,
    # num_train_epochs=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=1e-5,
    logging_steps=1,
    eval_strategy="epoch", # need to evaluate per epoch to see validataion if overfit
    save_strategy="epoch",
    load_best_model_at_end=True,
    remove_unused_columns=False,
)

#### Custom loss function
https://huggingface.co/docs/transformers/trainer_recipes#custom-loss-function

In [ ]:
# from transformers import Trainer
# class MyTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.pop("labels")
#         # remove pLDDT scores from input before training

#         # run forward pass with just sequences
#         outputs = model(**inputs)

#         # extract pLDDT from model output
#         predicted_plddt = outputs.plddt
#         mask = inputs["attention_mask"]

#         # calculate mean pLDDT per protein
#         masked_plddt = predicted_plddt * mask
#         sum_plddt = masked_plddt.sum(dim=1)
#         count_residues = mask.sum(dim=1)

#         # global predicted score for the whole protein
#         mean_predicted_plddt = sum_plddt / count_residues

#         # measure mean squared error of predicted_plddt v.s. knwon labels from baseline
#         loss = torch.nn.functional.mse_loss(mean_predicted_plddt, labels.float())

#         # logits = outputs[0]
#         # return my_custom_loss(logits, labels)
#         return (loss, outputs) if return_outputs else loss


from transformers import Trainer
import torch

class MyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        # remove pLDDT scores from input before training

        # run forward pass with just sequences
        outputs = model(**inputs)

        # extract pLDDT from model output
        predicted_plddt = outputs.plddt
        mask = inputs["attention_mask"]

        # match dimensions if needed
        if predicted_plddt.ndim == 3:
            predicted_plddt = predicted_plddt.mean(dim=-1)

        # calculate mean pLDDT per protein
        masked_plddt = predicted_plddt * mask
        sum_plddt = masked_plddt.sum(dim=1)
        count_residues = mask.sum(dim=1).clamp(min=1)

        # global predicted score for the whole protein
        mean_predicted_plddt = sum_plddt / count_residues

        # measure mean squared error of predicted_plddt v.s. knwon labels from baseline
        loss = torch.nn.functional.mse_loss(mean_predicted_plddt.float(), labels.float())

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            loss = self.compute_loss(model, inputs)

        # loss is a scalar tensor; detach it
        loss = loss.detach()

        return (loss, None, None)

In [ ]:
model.trunk.set_chunk_size(32)
# doing this because of the memory limit

# Run the training

In [ ]:
trainer = MyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # high confidence >= 0.7, 80%
    eval_dataset=test_dataset,  # high confidence >= 0.7, 20%
    # processing_class=tokenizer,
    # data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)


trainer.train()
# trainer.push_to_hub()

Epoch,Training Loss,Validation Loss
1,0.000026,0.000498
2,0.000012,0.000657
3,0.000013,0.000599
4,0.000010,0.000625
5,0.000019,0.000674
6,0.000003,0.000593
7,0.000009,0.000613
8,0.000003,0.000627
9,0.000009,0.000578
10,0.000014,0.000577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=80, training_loss=2.3380267703032586e-05, metrics={'train_runtime': 1806.9094, 'train_samples_per_second': 0.332, 'train_steps_per_second': 0.044, 'total_flos': 1624283373312000.0, 'train_loss': 2.3380267703032586e-05, 'epoch': 10.0})

In [ ]:
# Rerun structure prediction on low-confidence proteins using the fine-tuned model
# save new pdb to finetined_pdb_output_dir folder

import os
import torch

model.eval()
torch.cuda.empty_cache()

finetuned_pdb_dir = "finetuned_pdb_output_dir"
os.makedirs(finetuned_pdb_dir, exist_ok=True)

print("Low confidence proteins < 0.7")
print("Before length filter > 128 sequence length:", len(low_conf_df))

# Use only short sequences to match training setup avoid OOM
low_conf_eval_df = low_conf_df[low_conf_df["sequence"].str.len() <= 128].copy()
print("After length filter > 128 sequence length:", len(low_conf_eval_df))
print()
print("Low-confidence (< 0.7) proteins to rerun:", len(low_conf_eval_df))
count = 0

for _, row in low_conf_eval_df.iterrows():
    protein_id = row["protein_id"]
    sequence = row["sequence"]

    try:
        with torch.no_grad():
            pdb_string = model.infer_pdb(sequence)

        out_path = f"{finetuned_pdb_dir}/{protein_id}.pdb"
        with open(out_path, "w") as f:
            f.write(pdb_string)

        print(f"saved {out_path}")
        count += 1

    except Exception as e:
        print(f"failed on: {protein_id}: {e}")

print("Finished rerunning pdb generation on low-confidence set.")

Low confidence proteins < 0.7
Before length filter > 128 sequence length: 29
After length filter > 128 sequence length: 29

Low-confidence (< 0.7) proteins to rerun: 29
saved finetuned_pdb_output_dir/Ga0308414_100128614.pdb
saved finetuned_pdb_output_dir/Ga0308414_10002315.pdb
saved finetuned_pdb_output_dir/Ga0308414_10204732.pdb
saved finetuned_pdb_output_dir/Ga0308414_100001387.pdb
saved finetuned_pdb_output_dir/Ga0308414_10070522.pdb
saved finetuned_pdb_output_dir/Ga0308414_100000471.pdb
saved finetuned_pdb_output_dir/Ga0308414_100111113.pdb
saved finetuned_pdb_output_dir/Ga0308414_100224414.pdb
saved finetuned_pdb_output_dir/Ga0308414_100138218.pdb
saved finetuned_pdb_output_dir/Ga0308414_100063412.pdb
saved finetuned_pdb_output_dir/Ga0308414_1000001121.pdb
saved finetuned_pdb_output_dir/Ga0393409_050_1110_1301.pdb
saved finetuned_pdb_output_dir/Ga0308414_10001946.pdb
saved finetuned_pdb_output_dir/Ga0308414_10024823.pdb
saved finetuned_pdb_output_dir/Ga0308414_100002730.pdb
saved 

In [ ]:
# Recompute pLDDT scores from the new PDB files + Compare fine-tuned pLDDT vs baseline pLDDT

import os
import pandas as pd
import biotite.structure.io as bsio

finetuned_pdb_dir = "finetuned_pdb_output_dir"
finetuned_results = []

for filename in os.listdir(finetuned_pdb_dir):
    if filename.endswith(".pdb"):
        protein_id = filename.replace(".pdb", "")
        file_path = os.path.join(finetuned_pdb_dir, filename)

        try:
            struct = bsio.load_structure(file_path, extra_fields=["b_factor"])
            finetuned_plddt = float(struct.b_factor.mean())
            finetuned_results.append({
                "protein_id": protein_id,
                "finetuned_plddt_0_to_1": finetuned_plddt,
                "finetuned_plddt_0_to_100": finetuned_plddt * 100,
                "finetuned_pdb_file": file_path
            })

        except Exception as e:
            print(f"Error processing {filename}: {e}")

finetuned_df = pd.DataFrame(finetuned_results)
comparison_df = low_conf_eval_df.merge(
    finetuned_df,
    on="protein_id",
    how="inner"
)
comparison_df["plddt_change_0_to_1"] = (
    comparison_df["finetuned_plddt_0_to_1"] -
    comparison_df["baseline_plddt_0_to_1"]
)
comparison_df["plddt_change_0_to_100"] = (
    comparison_df["finetuned_plddt_0_to_100"] -
    comparison_df["baseline_plddt_0_to_100"]
)
comparison_df.to_csv("low_conf_finetuned_comparison.csv", index=False)
print(comparison_df[[
    "protein_id",
    "baseline_plddt_0_to_100",
    "finetuned_plddt_0_to_100",
    "plddt_change_0_to_100"
]])


# results
# print("\nAverage baseline pLDDT:", comparison_df["baseline_plddt_0_to_100"].mean())
# print("Average fine-tuned pLDDT:", comparison_df["finetuned_plddt_0_to_100"].mean())
# print("Average pLDDT change:", comparison_df["plddt_change_0_to_100"].mean())
# print("Saved to low_conf_finetuned_comparison.csv")

                  protein_id  baseline_plddt_0_to_100  \
0        Ga0308414_100128614                61.557813   
1         Ga0308414_10002315                26.217562   
2         Ga0308414_10204732                69.582011   
3        Ga0308414_100001387                54.850975   
4         Ga0308414_10070522                34.315488   
5        Ga0308414_100000471                49.091938   
6        Ga0308414_100111113                62.261128   
7        Ga0308414_100224414                67.854523   
8        Ga0308414_100138218                59.215251   
9        Ga0308414_100063412                68.786145   
10      Ga0308414_1000001121                60.425760   
11   Ga0393409_050_1110_1301                52.406814   
12        Ga0308414_10001946                66.082529   
13        Ga0308414_10024823                60.113145   
14       Ga0308414_100002730                43.684641   
15       Ga0308414_100070619                69.724800   
16       Ga0308414_100026712   

In [ ]:
print("==== SUMMARIZED RESULTS ====")
print("Num proteins:", len(comparison_df))
print("Avg baseline pLDDT:", round(comparison_df["baseline_plddt_0_to_100"].mean(), 2))
print("Avg fine-tuned pLDDT:", round(comparison_df["finetuned_plddt_0_to_100"].mean(), 2))
print("Avg improvement:", round(comparison_df["plddt_change_0_to_100"].mean(), 2))

improved = (comparison_df["plddt_change_0_to_100"] > 0).sum()
print("Num improved:", improved, "/", len(comparison_df))

==== SUMMARIZED RESULTS ====
Num proteins: 29
Avg baseline pLDDT: 56.22
Avg fine-tuned pLDDT: 56.2
Avg improvement: -0.02
Num improved: 14 / 29


**Conclusion (Data Subset II)**:
- fine-tuning led to moderate improvement of pLDDT score for 14 out of 29 proteins (evaluated using a larger dataset of sequences as compared to data subset I)
- But, average pLDDT score comparing the baseline to the finetuned model decreased by 0.02 on a pLDDT scale from 0-100.

- limited by memory and colab disk space, solutions:
  - smaller dataset
  - truncated sequences
  - lightweight training
- limited by speed of processor
  - cut down pdb files generated for baseline by about 75% (only run model on about 25% of the 440 sequences in original TASmania_hits_seqtk.faa)

During training:
- At the 1st epoch during training in fine-tuning, we can see that it has the lowest validation loss. Further epochs in the training overfit the model to the training data as we can see training loss decreases after the 1st epoch while validation loss increases.

Overall for both subset I and subset II datasets:

- Lightweight fine-tuning of Meta's ESMFold model slightly improved the pLDDT confidence sores on low-confidence proteins for subset I, but not subset II
- This improvement was consistent across most evaluated proteins  (10/10) for subset I, but only about half of the proteins for subset II (14/29)
- Both subset I and II are still about 25% of the original dataset
subset II included more sequences (about 4.6x more) for training and (about 3x more) for evaluation
- GPU memory limits in Google Collab required filtering protein sequences that were too long (over 128 amino acids) for training and evaluation

In the future, with more less memory limits, we could use:
- Larger Datasets
- Longer Protein Sequences


References:
  - https://huggingface.co/facebook/esmfold_v1
  - https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/protein_folding.ipynb
  - Code reference from:
https://github.com/facebookresearch/esm?tab=readme-ov-file#esmfold-structure-prediction-
  - https://github.com/facebookresearch/esm?tab=readme-ov-file#esmfold-structure-prediction-
  - https://github.com/huggingface/notebooks/blob/main/examples/protein_language_modeling.ipynb
  - https://huggingface.co/docs/transformers/training?training-args=logging
  - https://medium.com/@prabhatzade/freezing-layers-and-fine-tuning-transformer-models-in-pytorch-a-simple-guide-119cad0980c6
  - https://medium.com/we-talk-data/guide-to-freezing-layers-in-pytorch-best-practices-and-practical-examples-8e644e7a9598
  - https://huggingface.co/docs/transformers/trainer_recipes#custom-loss-function



- Acknowledgements:
  - Used chatGPT, Claude for questions on memory limits, suggested cutting data randomly, and troubleshooting to show training loss and validation loss. After trying this, we realized we could remove data that were longer than a set cutoff (128) instead
